In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import models
from torchvision.transforms import v2
import cv2
import os
import numpy as np

In [2]:
class ViolenceCNNPoseModel(nn.Module):
    def __init__(self, num_classes=2, dropout_rate=0.5):
        super().__init__()

        # -------- Enhanced Visual Backbone (ResNet34 for better features) --------
        resnet = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        self.cnn = nn.Sequential(*list(resnet.children())[:-1])
        self.visual_dim = 512

        # Strategic freezing: Train layer3 and layer4 for better adaptation
        for name, param in self.cnn.named_parameters():
            if "layer3" not in name and "layer4" not in name:
                param.requires_grad = False

        # -------- Enhanced Visual Temporal Modeling --------
        self.visual_lstm = nn.LSTM(
            input_size=512,
            hidden_size=256,
            num_layers=2,  # Deeper for better temporal understanding
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )

        # Visual Attention: Focus on important frames
        self.visual_attention = nn.Sequential(
            nn.Linear(512, 256),
            nn.Tanh(),
            nn.Linear(256, 1)
        )

        # -------- Enhanced Pose Branch --------
        self.pose_mlp = nn.Sequential(
            nn.Linear(34, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )

        self.pose_lstm = nn.LSTM(
            input_size=128,
            hidden_size=128,
            num_layers=2,  # Deeper for better temporal patterns
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )

        # Pose Attention: Focus on critical movements
        self.pose_attention = nn.Sequential(
            nn.Linear(256, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        # -------- Motion Feature Extraction --------
        # Captures frame differences (optical flow approximation)
        self.motion_conv = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        # -------- Enhanced Fusion --------
        fusion_dim = (256 * 2) + (128 * 2) + 128  # visual + pose + motion

        self.classifier = nn.Sequential(
            nn.LayerNorm(fusion_dim),
            nn.Linear(fusion_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)
        )

    def forward(self, video, pose):
        """
        Args:
            video: [B, 3, T, H, W] - Batch of video sequences
            pose:  [B, T, 34] - Batch of pose sequences (17 keypoints x 2 coords)
        
        Returns:
            logits: [B, num_classes] - Classification logits
        """
        B, C, T, H, W = video.shape

        # -------- Visual Branch with Attention --------
        video_frames = video.permute(0, 2, 1, 3, 4).reshape(B * T, 3, H, W)
        feat = self.cnn(video_frames)  # [B*T, 512, 1, 1]
        feat = feat.view(B, T, 512)

        # LSTM for temporal modeling
        vis_seq, _ = self.visual_lstm(feat)  # [B, T, 512]
        
        # Attention mechanism: weight frames by importance
        attn_weights = torch.softmax(self.visual_attention(vis_seq), dim=1)  # [B, T, 1]
        vis_feat = (vis_seq * attn_weights).sum(dim=1)  # [B, 512] - weighted sum instead of last timestep

        # -------- Pose Branch with Attention --------
        # Reshape for BatchNorm (needs [B*T, features])
        pose_reshaped = pose.reshape(B * T, 34)
        pose_feat = self.pose_mlp(pose_reshaped)
        pose_feat = pose_feat.view(B, T, 128)
        
        # LSTM for temporal pose patterns
        pose_seq, _ = self.pose_lstm(pose_feat)  # [B, T, 256]
        
        # Attention mechanism: weight poses by importance
        pose_attn_weights = torch.softmax(self.pose_attention(pose_seq), dim=1)  # [B, T, 1]
        pose_feat = (pose_seq * pose_attn_weights).sum(dim=1)  # [B, 256] - weighted sum

        # -------- Motion Features (Optical Flow Approximation) --------
        # Compute frame differences to detect rapid movements
        frame_diffs = video[:, :, 1:] - video[:, :, :-1]  # [B, 3, T-1, H, W]
        motion_feat = self.motion_conv(frame_diffs.mean(dim=2))  # [B, 128, 1, 1]
        motion_feat = motion_feat.view(B, -1)  # [B, 128]

        # -------- Multi-Modal Fusion --------
        fused = torch.cat([vis_feat, pose_feat, motion_feat], dim=1)  # [B, fusion_dim]

        return self.classifier(fused)

In [3]:
class ViolenceDataset(Dataset):
    def __init__(self, video_paths, labels, pose_dir, seq_len=16):
        self.video_paths = video_paths
        self.labels = labels
        self.pose_dir = pose_dir
        self.seq_len = seq_len

        self.transform = v2.Compose([
            v2.ToImage(),
            v2.Resize((112, 112)),  # correct for ResNet18 on CPU
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]

        # ---- Load video frames ----
        cap = cv2.VideoCapture(video_path)
        frames = []

        for _ in range(self.seq_len):
            ret, frame = cap.read()
            if not ret:
                break
            img = self.transform(frame)
            frames.append(img)

        cap.release()

        # Pad / handle short videos
        if len(frames) == 0:
            frames = [torch.zeros(3, 112, 112) for _ in range(self.seq_len)]

        while len(frames) < self.seq_len:
            frames.append(frames[-1].clone())

        video_tensor = torch.stack(frames).permute(1, 0, 2, 3)
        # shape: [3, T, H, W]

        # ---- Load precomputed pose ----
        video_name = os.path.splitext(os.path.basename(video_path))[0]
        pose_path = os.path.join(self.pose_dir, f"{video_name}.npy")

        if os.path.exists(pose_path):
            pose = torch.from_numpy(np.load(pose_path)).float()
        else:
            pose = torch.zeros(self.seq_len, 34)

        # Safety padding
        if pose.shape[0] < self.seq_len:
            pad = self.seq_len - pose.shape[0]
            pose = torch.cat([pose, pose[-1:].repeat(pad, 1)], dim=0)

        return video_tensor, pose, torch.tensor(self.labels[idx])


In [4]:
def get_dataset_lists(directory_map):
    all_videos, all_labels = [], []
    for path, label in directory_map.items():
        if os.path.exists(path):
            files = [os.path.join(path, f) for f in os.listdir(path) if f.lower().endswith(('.mp4', '.avi'))]
            all_videos.extend(files)
            all_labels.extend([label] * len(files))
            print(f"Added {len(files)} videos from {path}")
    return all_videos, all_labels

In [5]:
def train_model(model, train_loader, val_loader, epochs=15):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-3)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=True
    )

    best_acc = 0.0

    for epoch in range(epochs):
        # ---- TRAIN ----
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for videos, poses, labels in train_loader:
            videos, poses, labels = videos.to(device), poses.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(videos, poses)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        train_acc = 100. * correct / total
        print(f"Epoch {epoch+1}: Train Loss={running_loss/len(train_loader):.4f}, Train Acc={train_acc:.2f}%")

        # ---- VALIDATE ----
        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for videos, poses, labels in val_loader:
                videos, poses, labels = videos.to(device), poses.to(device), labels.to(device)
                outputs = model(videos, poses)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        val_acc = 100. * val_correct / val_total
        print(f"Validation Acc: {val_acc:.2f}%")

        # ---- SCHEDULER ----
        scheduler.step(val_acc)

        # ---- CHECKPOINT ----
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "best_violence_model_v3.pth")
            print("Checkpoint Saved!")


In [9]:
if __name__ == "__main__":
    # Your Dataset Map
    dataset_dirs = {
        # Non-Violence (0)
        # r"C:\Users\gowth\OneDrive\Desktop\mini project\datasets\Dataset\A-Dataset-for-Automatic-Violence-Detection-in-Videos-master\violence-detection-dataset\non-violent\cam1": 0,
        # r"C:\Users\gowth\OneDrive\Desktop\mini project\datasets\Dataset\A-Dataset-for-Automatic-Violence-Detection-in-Videos-master\violence-detection-dataset\non-violent\cam2": 0,
        # r"C:\Users\gowth\OneDrive\Desktop\mini project\datasets\Dataset\real life violence situations\Real Life Violence Dataset\NonViolence": 0,
        # r"C:\Users\gowth\OneDrive\Desktop\mini project\datasets\Dataset\SCVD\SCVD_converted\Train\Normal": 0,
        
        r"C:\Users\gowth\OneDrive\Desktop\mini project\datasets\Dataset\RWF-2000\train\NonFight":0,


        # Violence (1)
        # r"C:\Users\gowth\OneDrive\Desktop\mini project\datasets\Dataset\A-Dataset-for-Automatic-Violence-Detection-in-Videos-master\violence-detection-dataset\violent\cam1": 1,
        # r"C:\Users\gowth\OneDrive\Desktop\mini project\datasets\Dataset\A-Dataset-for-Automatic-Violence-Detection-in-Videos-master\violence-detection-dataset\violent\cam2": 1,
        # r"C:\Users\gowth\OneDrive\Desktop\mini project\datasets\Dataset\real life violence situations\Real Life Violence Dataset\Violence": 1,
        # r"C:\Users\gowth\OneDrive\Desktop\mini project\datasets\Dataset\SCVD\SCVD_converted\Train\Violence": 1,
        # r"C:\Users\gowth\OneDrive\Desktop\mini project\datasets\Dataset\SCVD\SCVD_converted\Train\Weaponized": 1,
        
        r"C:\Users\gowth\OneDrive\Desktop\mini project\datasets\Dataset\RWF-2000\train\Fight":1,
    }

    # 1. Load lists
    vids, lbls = get_dataset_lists(dataset_dirs)
    
    # 2. Create Dataset & Split (80% Train, 20% Val)
    dataset = ViolenceDataset(
    vids,
    lbls,
    pose_dir=r"C:\path\to\saved_poses",
    seq_len=16
)

    train_size = int(0.9 * len(dataset))
    val_size = len(dataset) - train_size
    train_ds, val_ds = random_split(dataset, [train_size, val_size])

    # 3. DataLoaders
    train_loader = DataLoader(train_ds, batch_size=10, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=5, shuffle=False)

    # 4. Run Training
    model = ViolenceCNNPoseModel()

    model.load_state_dict(
        torch.load("best_violence_model_v3.pth", map_location="cpu")
    )

    train_model(model, train_loader, val_loader, epochs=25)

Added 800 videos from C:\Users\gowth\OneDrive\Desktop\mini project\datasets\Dataset\RWF-2000\train\NonFight
Added 788 videos from C:\Users\gowth\OneDrive\Desktop\mini project\datasets\Dataset\RWF-2000\train\Fight


C:\Users\gowth\AppData\Local\Temp\ipykernel_27776\1664610081.py:46: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load("best_violence_model_v3.pth", map_location="cpu"

Epoch 1: Train Loss=0.6904, Train Acc=63.19%
Validation Acc: 69.18%
Checkpoint Saved!
Epoch 2: Train Loss=0.6024, Train Acc=67.74%
Validation Acc: 71.70%
Checkpoint Saved!
Epoch 3: Train Loss=0.5818, Train Acc=69.49%
Validation Acc: 69.18%
Epoch 4: Train Loss=0.5855, Train Acc=69.21%
Validation Acc: 72.96%
Checkpoint Saved!
Epoch 5: Train Loss=0.5936, Train Acc=70.26%
Validation Acc: 69.18%
Epoch 6: Train Loss=0.5551, Train Acc=72.08%
Validation Acc: 76.10%
Checkpoint Saved!
Epoch 7: Train Loss=0.5623, Train Acc=72.50%
Validation Acc: 73.58%
Epoch 8: Train Loss=0.5535, Train Acc=72.08%
Validation Acc: 74.21%
Epoch 9: Train Loss=0.5351, Train Acc=74.67%
Validation Acc: 74.84%
Epoch 10: Train Loss=0.4762, Train Acc=79.08%
Validation Acc: 74.21%
Epoch 11: Train Loss=0.4613, Train Acc=78.66%
Validation Acc: 75.47%
Epoch 12: Train Loss=0.4518, Train Acc=77.82%
Validation Acc: 74.21%
Epoch 13: Train Loss=0.4139, Train Acc=80.41%
Validation Acc: 77.36%
Checkpoint Saved!
Epoch 14: Train Loss=0

In [7]:
import torch

# Check if CUDA (GPU support) is available
if torch.cuda.is_available():
    print("Yes ✅, GPU is available!")
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("No ❌, GPU not detected. Using CPU.")


Yes ✅, GPU is available!
GPU Name: NVIDIA GeForce RTX 3050 Laptop GPU
